# Notebook 2: Image Augmentation & Baseline Model

## Overview

This notebook explores **data augmentation** techniques and their effect on model training:

1. **Loading preprocessed datasets** from Notebook 1
2. **Data augmentation techniques** - RandomFlip, RandomRotation, RandomContrast
3. **Building a modular CNN pipeline** - Separating augmentation, preprocessing, feature extraction, and classification
4. **Training and evaluating** the augmented baseline model
5. **Confusion matrix analysis** to understand where the model fails

### Key Insight

Data augmentation creates *artificial variations* of training images (rotations, flips, contrast changes) so the model sees more diverse examples. This helps prevent overfitting and improves generalization. However, augmentation alone cannot compensate for a model with limited capacity - that's where **transfer learning** (Notebook 3) comes in.

### Prerequisites

- Run **Notebook 1** first to generate the processed datasets in `data/images/processed/`

---

---
## 1.&nbsp; Import Libraries

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import confusion_matrix
from tensorflow import Tensor, concat
from tensorflow.data import Dataset
from tensorflow.keras import Sequential, layers
from tensorflow.keras.callbacks import EarlyStopping, History


## 2.&nbsp; Loading, Batching, and Prefetching Datasets 🌩

In [ ]:
#@title Defining Dataset Loading Function (from previous notebook)
PROCESSED_DIR: str = "../data/images/processed"

def load_datasets(
    processed_dir: str,
    batch_size: int = 32
) -> dict[str, Dataset]:
    """
    Loads, batches, and prefetches processed data. Returns in dictionary with tf.data.Dataset objects for each split.

    Args:
        processed_dir (str): Path to directory with dir/split/Dataset structure
        batch_size (int): Batch size

    Returns:
        dict: mapping of split names to their respective tf.data.Dataset objects

    Examples:
    >>> datasets = load_datasets("../data/images/processed/", batch_size=32)
    >>> model.fit(
    ...     datasets['train'],
    ...     validation_data=datasets['val'],
    ...     epochs=10
    ... )
    """
    # Identify splits present in processed directory
    splits: list[str] = [
        s for s in os.listdir(processed_dir)
        if os.path.isdir(os.path.join(processed_dir, s))
    ]

    # Initialize autotune for prefetching
    autotune = tf.data.AUTOTUNE

    # Initialize dict that will be returned
    datasets: dict[str, Dataset] = {}

    for split in splits:
        # Load each dataset
        path = os.path.join(processed_dir, split)
        ds: Dataset = Dataset.load(path)
        # Batch, cache, and prefetch for optimal performance
        datasets[split]  = (
            ds
            .batch(batch_size)
            .cache()
            .prefetch(buffer_size=autotune)
        )

    return datasets

In [ ]:
BATCH_SIZE: int = 32

# Load saved Dataset objects, batching and prefetching them
datasets: dict[str, Dataset] = load_datasets(
    processed_dir=PROCESSED_DIR,
    batch_size=BATCH_SIZE,
)

---
## 3.&nbsp; Image Augmentation ✏️

When training image classifiers with TensorFlow, you typically insert a small data augmentation block into your model pipeline to deliberately introduce variation and help the network generalise. In Keras, you'd create something like:

In [ ]:
RANDOM_SEED: int = 42

data_augmentor: Sequential = Sequential([
    layers.RandomFlip("horizontal", seed=RANDOM_SEED),
    layers.RandomRotation(0.9, seed=RANDOM_SEED),
    layers.RandomContrast(factor=1, seed=RANDOM_SEED),
], name='image_augmentor')

This sequence applies a random horizontal flip, a slight rotation , and a contrast adjustment during training. These are all part of the built‑in image augmentation layers in Keras, which are active only at training time and bypassed at inference. You can chain other augmentations too, such as zoom, crop or brightness, you can find them in the [tensorflow docs for layers](https://www.tensorflow.org/api_docs/python/tf/keras/layers) or in [Keras' own documentation](https://keras.io/api/layers/preprocessing_layers/image_augmentation/).

To give you an idea of how these layers affect the images, let's take a look at a handful of outputs:

In [ ]:
plt.figure(figsize=(10, 10))
for images, _ in datasets['train'].take(1):
    augmented_images: Tensor = data_augmentor(images)
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(augmented_images[i].numpy().astype("uint8"))
        plt.axis("off")

As you can see, each image has been randomly flipped horizontally, rotated slightly, or had its contrast changed — sometimes all at once.

---
## 3.&nbsp; Adding the Image Augmentation Layers to the Model 🚀

As model pipelines get larger it's good practice to compartmentalize different functionality. Keras makes this easy by allowing us to nest keras.Sequential objects inside each other

In [ ]:
preprocessor: Sequential = Sequential([
    layers.Rescaling(1./255),
], name='preprocessor')

feature_extractor: Sequential = Sequential([

    # Convolution Block 1
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Convolution Block 2
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Convolution Block 3
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),
], name='feature_extractor')

classifier: Sequential = Sequential([
    layers.Flatten(),

    # Hidden dense layer
    layers.Dense(128, activation="relu"),

    # Output layer
    layers.Dense(3, activation="softmax"),
], name='classifier')

Now we can combine the parts together.

Let's build one model with and one without augmentation:

In [ ]:
model: Sequential = Sequential([
    layers.Input((256, 256, 3)),
    data_augmentor,
    preprocessor,
    feature_extractor,
    classifier,
], name='full_pipeline')
model.summary()

In [ ]:
# Initializing early stopping
es = EarlyStopping(
    monitor="val_loss",
    patience=20,
    verbose=1,
    restore_best_weights=True,
)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

In [ ]:
history: History = model.fit(
    datasets['train'], # <--- notice how we can access the different splits from the datasets dict
    validation_data=datasets['val'], # <--- notice how we can access the different splits from the datasets dict
    epochs=50,
    #callbacks=[es],
    verbose=1,
)

Awesome! Now that we have out model, let's take a look at the training curve.

In [ ]:
acc: list[float] = history.history['accuracy']
val_acc: list[float] = history.history['val_accuracy']
loss: list[float] = history.history['loss']
val_loss: list[float] = history.history['val_loss']
epochs_range: range = range(len(acc))

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(epochs_range, acc, label='Train')
plt.plot(epochs_range, val_acc, label='Val')
plt.title('Accuracy')
plt.legend()

plt.subplot(1,2,2)
plt.plot(epochs_range, loss, label='Train')
plt.plot(epochs_range, val_loss, label='Val')
plt.title('Loss')
plt.legend()
plt.show()

When using image augmentation, you're likely to see the model take more epochs to learn because the task it's been given is much harder. But, if you've got a good pipeline, you're likely to see validation performance increase for longer as the model is forced to learn something more general in order to perform well on this more diverse dataset.

---
## 4.&nbsp; Evaluating the model 🔎

Let's have a look at the test set and see where our model suceeds and fails.

In [ ]:
model.evaluate(datasets['test'])

In [ ]:
class_names: list[str] = ['A', 'B', 'C']

# Collect all images and labels from the test set
all_images: list[Tensor] = []
all_labels: list[Tensor] = []
for images, labels in datasets['test']:
    all_images.append(images)
    all_labels.append(labels)

all_images: Tensor = concat(all_images, axis=0)
all_labels: Tensor = concat(all_labels, axis=0)

preds: np.ndarray = model.predict(all_images)
pred_labels: np.ndarray = np.argmax(preds, axis=1)

num_images: int = len(all_images)

# Calculate the number of rows and columns for the subplot grid
n_cols: int = 5  # You can adjust the number of columns as needed
n_rows: int = (num_images + n_cols - 1) // n_cols # Calculate rows needed

plt.figure(figsize=(n_cols * 2.5, n_rows * 3)) # Adjust figure size based on grid size
for i in range(num_images):
    plt.subplot(n_rows, n_cols, i + 1)
    plt.imshow(all_images[i].numpy().astype("uint8"))
    true_label: str = class_names[all_labels[i]]
    predicted_label: str = class_names[pred_labels[i]]
    plt.title(
        f"True: {true_label}\nPred: {predicted_label}",
        color='green' if true_label == predicted_label else 'red')
    plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Create the confusion matrix
cm: np.ndarray = confusion_matrix(all_labels, pred_labels)

# Plot the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

---
## 5.&nbsp; Challenge 🏆
Incorporate image augmentation into your pipeline! See if this extra diversity in the training data successfully encourages the model learn a pattern that generalizes

---
## Summary

In this notebook, we:

1. **Applied data augmentation** (RandomFlip, RandomRotation, RandomContrast) to artificially expand our training set
2. **Built a modular pipeline** separating augmentation, preprocessing, feature extraction, and classification
3. **Trained and evaluated** the augmented model with confusion matrix analysis

### Key Takeaway

Data augmentation helps prevent overfitting, but **a baseline CNN has limited capacity** to learn complex features. The ~62% accuracy shows that augmentation alone is not enough - the model architecture itself needs to be more powerful. This is exactly what **transfer learning** solves.

### Next Step

Proceed to **[Notebook 3: Transfer Learning](03_transfer_learning.ipynb)** to see how pre-trained models achieve 99%+ accuracy.